In [ ]:
# Chạy local - không cần mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# **Code cắt 4 góc của bouding box**

In [ ]:
import cv2
import os
import glob
import random
import numpy as np

# --- CẤU HÌNH ---
BRACKET_CLASS = 13  # Nhãn của mắc cài cần loại bỏ
COLOR_GREEN = (0, 255, 0)    # Vùng Label 0
COLOR_RED   = (0, 0, 255)    # Vùng Label 1
COLOR_WHITE = (255, 255, 255) # Màu của Răng
PADDING_PX = 10 # Số pixel mở rộng cho khung răng khi vẽ (để không che các box trong)

def yolo_to_pixel(yolo_box, img_w, img_h):
    x_c, y_c, w, h = yolo_box
    x1 = int((x_c - w / 2) * img_w)
    y1 = int((y_c - h / 2) * img_h)
    x2 = int((x_c + w / 2) * img_w)
    y2 = int((y_c + h / 2) * img_h)
    return x1, y1, x2, y2

def pixel_to_yolo(pixel_box, img_w, img_h):
    x1, y1, x2, y2 = pixel_box
    w = x2 - x1
    h = y2 - y1
    x_c = x1 + w / 2
    y_c = y1 + h / 2
    return x_c / img_w, y_c / img_h, w / img_w, h / img_h

def get_iou_containment(box_inner, box_outer):
    ix1, iy1, ix2, iy2 = box_inner
    ox1, oy1, ox2, oy2 = box_outer
    center_x = (ix1 + ix2) / 2
    center_y = (iy1 + iy2) / 2
    return ox1 < center_x < ox2 and oy1 < center_y < oy2

def process_batch(img_folder, lbl_folder, out_img_folder, out_lbl_folder):
    os.makedirs(out_img_folder, exist_ok=True)
    os.makedirs(out_lbl_folder, exist_ok=True)

    img_files = glob.glob(os.path.join(img_folder, "*"))
    img_files = [f for f in img_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print(f"Tìm thấy {len(img_files)} ảnh. Bắt đầu xử lý...")

    for img_path in img_files:
        filename = os.path.basename(img_path)
        base_name = os.path.splitext(filename)[0]
        lbl_path = os.path.join(lbl_folder, base_name + ".txt")

        # 1. Đọc ảnh
        img = cv2.imread(img_path)
        if img is None: continue
        img_h, img_w = img.shape[:2]

        # 2. Đọc nhãn gốc
        brackets = []
        teeth = []
        if not os.path.exists(lbl_path): continue
        with open(lbl_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = list(map(float, line.strip().split()))
                if len(parts) < 5: continue
                cls = int(parts[0])
                bbox = parts[1:5]
                coords = yolo_to_pixel(bbox, img_w, img_h)
                if cls == BRACKET_CLASS: brackets.append(coords)
                else: teeth.append({'coords': coords, 'id': cls})

        # 3. Xử lý logic và tạo dữ liệu mới
        new_labels = []

        for tooth in teeth:
            t_box = tooth['coords'] # Tọa độ GỐC của răng
            t_id = tooth['id']
            tx1, ty1, tx2, ty2 = t_box

            # --- BƯỚC 1: LƯU THÔNG TIN GỐC CỦA RĂNG VÀO FILE LABEL ---
            # Sử dụng t_box gốc để lưu
            yolo_t = pixel_to_yolo(t_box, img_w, img_h)
            new_labels.append(f"{t_id} {yolo_t[0]:.6f} {yolo_t[1]:.6f} {yolo_t[2]:.6f} {yolo_t[3]:.6f}")

            # --- BƯỚC 2: VẼ 4 VÙNG MÀU BÊN TRONG (NẾU CÓ) ---
            matching_bracket = None
            for b_box in brackets:
                if get_iou_containment(b_box, t_box):
                    matching_bracket = b_box
                    break

            if matching_bracket:
                bx1, by1, bx2, by2 = matching_bracket
                rois = {
                    'G': (tx1, ty1, tx2, by1), 'I': (tx1, by2, tx2, ty2),
                    'M': (tx1, by1, bx1, by2), 'D': (bx2, by1, tx2, by2)
                }
                for roi_coords in rois.values():
                    random_cls = random.choice([0, 1])
                    color = COLOR_GREEN if random_cls == 0 else COLOR_RED

                    # Vẽ các vùng màu (độ dày 2)
                    cv2.rectangle(img, (roi_coords[0], roi_coords[1]), (roi_coords[2], roi_coords[3]), color, 2)

                    yolo_roi = pixel_to_yolo(roi_coords, img_w, img_h)
                    new_labels.append(f"{random_cls} {yolo_roi[0]:.6f} {yolo_roi[1]:.6f} {yolo_roi[2]:.6f} {yolo_roi[3]:.6f} {t_id}")

            # --- BƯỚC 3: TÍNH TOÁN VÀ VẼ KHUNG RĂNG MỞ RỘNG (ĐỂ HIỂN THỊ) ---
            # Tạo tọa độ vẽ mở rộng thêm PADDING_PX
            draw_tx1 = max(0, tx1 - PADDING_PX)
            draw_ty1 = max(0, ty1 - PADDING_PX)
            draw_tx2 = min(img_w, tx2 + PADDING_PX)
            draw_ty2 = min(img_h, ty2 + PADDING_PX)

            # Vẽ khung trắng bao quanh bên ngoài (độ dày 2)
            cv2.rectangle(img, (draw_tx1, draw_ty1), (draw_tx2, draw_ty2), COLOR_WHITE, 2)

        # 4. Lưu kết quả
        cv2.imwrite(os.path.join(out_img_folder, filename), img)
        with open(os.path.join(out_lbl_folder, base_name + ".txt"), 'w') as f:
            f.write('\n'.join(new_labels))

    print(f"Hoàn tất! Box răng màu trắng được vẽ rộng hơn {PADDING_PX}px để bao bọc các vùng màu.")

# --- CHẠY CHƯƠNG TRÌNH LOCAL ---
# Thay đổi đường dẫn phù hợp với môi trường local
folder_anh_goc = './data/images'           # Thư mục chứa ảnh gốc
folder_label_goc = './data/labels'         # Thư mục chứa label gốc
folder_anh_ra = './data/results/images'    # Thư mục lưu ảnh đã xử lý
folder_label_ra = './data/results/labels'  # Thư mục lưu label đã xử lý

if __name__ == "__main__":
    if os.path.exists(folder_anh_goc) and os.path.exists(folder_label_goc):
        process_batch(folder_anh_goc, folder_label_goc, folder_anh_ra, folder_label_ra)
    else:
        print(f"Đường dẫn folder input không tồn tại.")
        print(f"Tạo folder: {folder_anh_goc} và {folder_label_goc}")
        os.makedirs(folder_anh_goc, exist_ok=True)
        os.makedirs(folder_label_goc, exist_ok=True)
        print("Hãy copy ảnh và labels vào các folder trên, sau đó chạy lại.")

Tìm thấy 9 ảnh. Bắt đầu xử lý...
Hoàn tất! Box răng màu trắng được vẽ rộng hơn 10px để bao bọc các vùng màu.


# **Code argumentation**

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path

# Cài đặt trước: pip install albumentations opencv-python
import albumentations as A

def read_yolo_annotation(txt_path):
    """Đọc file annotation YOLO format"""
    boxes = []
    with open(txt_path, 'r') as f:
        for line in f.readlines():
            data = line.strip().split()
            if len(data) >= 5:
                class_id = int(data[0])
                x_center, y_center, width, height = map(float, data[1:5])
                # Kiểm tra nếu có trường thông tin thêm (giá trị thứ 6)
                extra_info = int(data[5]) if len(data) >= 6 else None
                boxes.append([class_id, x_center, y_center, width, height, extra_info])
    return boxes

def write_yolo_annotation(txt_path, boxes):
    """Ghi file annotation YOLO format"""
    with open(txt_path, 'w') as f:
        for box in boxes:
            class_id, x_center, y_center, width, height, extra_info = box
            if extra_info is not None:
                f.write(f"{int(class_id)} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f} {int(extra_info)}\n")
            else:
                f.write(f"{int(class_id)} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

def yolo_to_pascal(boxes, img_width, img_height):
    """Chuyển từ YOLO format (x_center, y_center, width, height) sang Pascal VOC (x_min, y_min, x_max, y_max)"""
    pascal_boxes = []
    class_labels = []
    extra_infos = []

    for box in boxes:
        class_id, x_center, y_center, width, height, extra_info = box

        x_min = (x_center - width / 2) * img_width
        y_min = (y_center - height / 2) * img_height
        x_max = (x_center + width / 2) * img_width
        y_max = (y_center + height / 2) * img_height

        pascal_boxes.append([x_min, y_min, x_max, y_max])
        class_labels.append(class_id)
        extra_infos.append(extra_info)

    return pascal_boxes, class_labels, extra_infos

def pascal_to_yolo(boxes, class_labels, extra_infos, img_width, img_height):
    """Chuyển từ Pascal VOC sang YOLO format"""
    yolo_boxes = []

    for box, class_id, extra_info in zip(boxes, class_labels, extra_infos):
        x_min, y_min, x_max, y_max = box

        x_center = ((x_min + x_max) / 2) / img_width
        y_center = ((y_min + y_max) / 2) / img_height
        width = (x_max - x_min) / img_width
        height = (y_max - y_min) / img_height

        # Đảm bảo giá trị trong khoảng [0, 1]
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        width = max(0, min(1, width))
        height = max(0, min(1, height))

        yolo_boxes.append([class_id, x_center, y_center, width, height, extra_info])

    return yolo_boxes

def augment_image_and_boxes(image, boxes, augmentation_name):
    """Áp dụng augmentation cho ảnh và boxes"""
    img_height, img_width = image.shape[:2]

    # Chuyển đổi YOLO sang Pascal VOC
    pascal_boxes, class_labels, extra_infos = yolo_to_pascal(boxes, img_width, img_height)

    # Định nghĩa các augmentation
    if augmentation_name == "rotate_left":
        transform = A.Compose([
            A.Rotate(limit=(-15, -15), p=1.0)
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    elif augmentation_name == "rotate_right":
        transform = A.Compose([
            A.Rotate(limit=(15, 15), p=1.0)
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    elif augmentation_name == "flip":
        transform = A.Compose([
            A.HorizontalFlip(p=1.0)
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    elif augmentation_name == "brightness_up":
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(0.2, 0.2), contrast_limit=0, p=1.0)
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    elif augmentation_name == "brightness_down":
        transform = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(-0.2, -0.2), contrast_limit=0, p=1.0)
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

    # Áp dụng augmentation
    transformed = transform(image=image, bboxes=pascal_boxes, class_labels=class_labels)

    aug_image = transformed['image']
    aug_boxes = transformed['bboxes']
    aug_labels = transformed['class_labels']

    # Chuyển đổi lại sang YOLO format (giữ nguyên extra_infos)
    yolo_boxes = pascal_to_yolo(aug_boxes, aug_labels, extra_infos, img_width, img_height)

    return aug_image, yolo_boxes

def augment_dataset(image_folder, annotation_folder, output_image_folder, output_annotation_folder):
    """Thực hiện augmentation cho toàn bộ dataset"""

    # Tạo thư mục output chính
    os.makedirs(output_image_folder, exist_ok=True)
    os.makedirs(output_annotation_folder, exist_ok=True)

    # Danh sách các augmentation
    augmentations = ["rotate_left", "rotate_right", "flip", "brightness_up", "brightness_down"]

    # Lấy danh sách file ảnh
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    image_files = []
    for ext in image_extensions:
        image_files.extend(Path(image_folder).glob(f'*{ext}'))
        image_files.extend(Path(image_folder).glob(f'*{ext.upper()}'))

    print(f"Tìm thấy {len(image_files)} ảnh")

    for img_path in image_files:
        img_name = img_path.stem
        img_ext = img_path.suffix

        # Tạo folder con cho từng ảnh
        img_output_folder = Path(output_image_folder) / img_name
        ann_output_folder = Path(output_annotation_folder) / img_name
        os.makedirs(img_output_folder, exist_ok=True)
        os.makedirs(ann_output_folder, exist_ok=True)

        # Đường dẫn file annotation
        ann_path = Path(annotation_folder) / f"{img_name}.txt"

        if not ann_path.exists():
            print(f"Không tìm thấy annotation cho {img_name}, bỏ qua...")
            continue

        # Đọc ảnh và annotation
        image = cv2.imread(str(img_path))
        if image is None:
            print(f"Không thể đọc ảnh {img_path}, bỏ qua...")
            continue

        boxes = read_yolo_annotation(ann_path)

        # Copy ảnh và annotation gốc vào folder con
        output_img_path = img_output_folder / f"{img_name}_original{img_ext}"
        output_ann_path = ann_output_folder / f"{img_name}_original.txt"
        cv2.imwrite(str(output_img_path), image)
        write_yolo_annotation(output_ann_path, boxes)

        print(f"\nXử lý ảnh: {img_name}")
        print(f"  - Lưu bản gốc: {img_name}_original{img_ext}")

        # Áp dụng các augmentation
        for aug_name in augmentations:
            try:
                aug_image, aug_boxes = augment_image_and_boxes(image, boxes, aug_name)

                # Lưu ảnh và annotation đã augment vào folder con
                aug_img_name = f"{img_name}_{aug_name}{img_ext}"
                aug_ann_name = f"{img_name}_{aug_name}.txt"

                aug_img_path = img_output_folder / aug_img_name
                aug_ann_path = ann_output_folder / aug_ann_name

                cv2.imwrite(str(aug_img_path), aug_image)
                write_yolo_annotation(aug_ann_path, aug_boxes)

                print(f"  - Tạo: {aug_img_name}")

            except Exception as e:
                print(f"  - Lỗi khi augment với {aug_name}: {str(e)}")

    print("\n" + "="*60)
    print("Hoàn thành!")
    print(f"Kết quả được lưu tại:")
    print(f"  - Ảnh: {output_image_folder}")
    print(f"  - Annotations: {output_annotation_folder}")
    print(f"  - Mỗi ảnh gốc có 1 folder con chứa 6 ảnh (1 gốc + 5 augmented)")
    print("="*60)

# ===== SỬ DỤNG LOCAL =====
if __name__ == "__main__":
    # Cấu hình đường dẫn local
    IMAGE_FOLDER = "./data/results/images"           # Thư mục chứa ảnh đã xử lý (từ code cắt 4 góc)
    ANNOTATION_FOLDER = "./data/results/labels"      # Thư mục chứa annotation đã xử lý
    OUTPUT_IMAGE_FOLDER = "./data/augmented/images"  # Thư mục lưu ảnh đã augment
    OUTPUT_ANNOTATION_FOLDER = "./data/augmented/labels"  # Thư mục lưu annotation đã augment

    # Kiểm tra folder tồn tại
    if not os.path.exists(IMAGE_FOLDER):
        print(f"Folder không tồn tại: {IMAGE_FOLDER}")
        print("Hãy chạy code cắt 4 góc trước (Cell 3)")
    elif not os.path.exists(ANNOTATION_FOLDER):
        print(f"Folder không tồn tại: {ANNOTATION_FOLDER}")
        print("Hãy chạy code cắt 4 góc trước (Cell 3)")
    else:
        # Thực hiện augmentation
        augment_dataset(IMAGE_FOLDER, ANNOTATION_FOLDER, OUTPUT_IMAGE_FOLDER, OUTPUT_ANNOTATION_FOLDER)

Đang cài đặt thư viện...
Tìm thấy 9 ảnh

Xử lý ảnh: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f
  - Lưu bản gốc: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f_original.jpg
  - Tạo: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f_rotate_left.jpg
  - Tạo: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f_rotate_right.jpg
  - Tạo: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f_flip.jpg
  - Tạo: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f_brightness_up.jpg
  - Tạo: 4_JPG.rf.9aea4229131ac57fa7448b0b078fdf0f_brightness_down.jpg

Xử lý ảnh: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31
  - Lưu bản gốc: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31_original.jpg
  - Tạo: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31_rotate_left.jpg
  - Tạo: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31_rotate_right.jpg
  - Tạo: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31_flip.jpg
  - Tạo: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31_brightness_up.jpg
  - Tạo: 8_JPG.rf.115ebc59d042e2db2a85add547b0da31_brightness_down.jpg

Xử lý ảnh: 2_JPG.rf.569e48ee4aadfe0b3